# When Does Graph Structure Help in MARL? --- Colab runner

This notebook reproduces every phase of the paper *When Does Graph Structure Help in Multi-Agent Reinforcement Learning?* in one place.

**What it does** (≈30--90 min wall on Colab CPU, ≈15--40 min on T4 GPU):

1. Clone the repo at the desired branch (default `phase-1-pilot`).
2. Install dependencies.
3. Run the unit-test suite as a smoke check.
4. Run each phase (1, 2, 4) sequentially. **Toggle phases in the *Run config* cell.**
5. Generate figures + summary tables under `results/<phase>/`.
6. Compile `paper/main.pdf` with the real results inserted.
7. Zip everything (results + paper PDF) into `gnnmarl_outputs.zip` and download it.

**Why use Colab?** The model is tiny (~30k params); GPU offers a modest 2--3x speedup over Colab CPU for this workload. The bigger reason is convenience and reproducibility.

**After the run**, unzip `gnnmarl_outputs.zip` into the repo's `results/` directory on your local machine; downstream analysis scripts and the paper build pick it up automatically.

## 1. Setup

Clone the repo and install the package. Change `BRANCH` if you want a different branch (e.g. `main` once Phase 1 is merged).

In [ ]:
REPO_URL = 'https://github.com/Evasion-OC/when-graphs-help-marl.git'
BRANCH   = 'phase-1-pilot'   # change if needed

import os, subprocess, sys
if not os.path.exists('when-graphs-help-marl'):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL])
os.chdir('when-graphs-help-marl')
print('cwd:', os.getcwd())
subprocess.check_call(['git', 'log', '--oneline', '-1'])


In [ ]:
!pip install -q -e '.[dev]' 2>&1 | tail -5


## 2. Smoke check

Run the full unit + integration test suite. Should report ~71 passed.

In [ ]:
!PYTHONIOENCODING=utf-8 python -m pytest -q 2>&1 | tail -5


## 3. GPU / device check

Colab gives you either a T4 GPU or CPU-only depending on the runtime type (Runtime > Change runtime type). The trainer auto-detects.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))


## 4. Run config

Toggle which phases to run. Defaults run all three reportable phases at their scaled budgets (matches what the paper reports).

If Colab disconnects mid-run, just re-run the affected phase cell; existing CSVs under `results/<phase>/<run_id>/episodes.csv` are not overwritten unless you clear that directory.

In [ ]:
RUN_PHASE_1 = True   # pilot: 4 algos x 3 seeds x 50k steps  (~30-90 min)
RUN_PHASE_2 = True   # E1 sweep: 4 algos x 2 N x 2 graphs x 3 seeds x 30k  (~50-120 min)
RUN_PHASE_4 = True   # depth x diameter ablation: 3 graphs x 4 depths x 3 seeds x 20k  (~30-90 min)


## 5. Phase 1 --- pilot

Trains each of {IQL, VDN, QMIX, GNN-QMIX} for 3 seeds x 50k env steps on CoordGrid ($N=4$, ring). Per-run wall is ~3--6 min depending on hardware.

In [ ]:
if RUN_PHASE_1:
    !PYTHONIOENCODING=utf-8 python scripts/phase1_pilot.py 2>&1 | tee results/phase1_log.txt
    !PYTHONIOENCODING=utf-8 python scripts/phase1_analysis.py 2>&1 | tail -20
else:
    print('skipped phase 1')


## 6. Phase 2 --- E1 sweep (H1, H2)

$N \in \{4, 8\} \times \{$ring, Erdős--Rényi (matched density)$\} \times$ 4 algos $\times$ 3 seeds at $3\times 10^4$ env steps per run. 48 runs total.

In [ ]:
if RUN_PHASE_2:
    !PYTHONIOENCODING=utf-8 python scripts/phase2_e1_sweep.py 2>&1 | tee results/phase2_log.txt
    !PYTHONIOENCODING=utf-8 python scripts/phase2_analysis.py 2>&1 | tail -30
else:
    print('skipped phase 2')


## 7. Phase 4 --- depth $\times$ diameter ablation (H3)

GNN depth $L \in \{1,2,3,4\}$ crossed with graph diameter $d \in \{1,2,3\}$ ($N=4$ fixed), 3 seeds, 20k env steps per run. 36 runs total.

In [ ]:
if RUN_PHASE_4:
    !PYTHONIOENCODING=utf-8 python scripts/phase4_ablation.py 2>&1 | tee results/phase4_log.txt
    !PYTHONIOENCODING=utf-8 python scripts/phase4_analysis.py 2>&1 | tail -20
else:
    print('skipped phase 4')


## 8. Build the paper

Regenerates the LaTeX inserts from the result CSVs and compiles `paper/main.pdf`. Requires `texlive` --- Colab usually has it but the install line is here just in case.

In [ ]:
import subprocess
try:
    subprocess.check_call(['which', 'pdflatex'])
    have_latex = True
except subprocess.CalledProcessError:
    have_latex = False
if not have_latex:
    !apt-get -qq install -y texlive-latex-extra texlive-fonts-recommended texlive-science 2>&1 | tail -3


In [ ]:
!PYTHONIOENCODING=utf-8 python scripts/build_paper_inserts.py
%cd paper
!pdflatex -interaction=nonstopmode main.tex > /tmp/lp.txt 2>&1 ; bibtex main > /tmp/bt.txt 2>&1 ; pdflatex -interaction=nonstopmode main.tex > /tmp/lp.txt 2>&1 ; pdflatex -interaction=nonstopmode main.tex 2>&1 | tail -3
%cd ..


## 9. Bundle + download

Zips the `results/` directory and the compiled `paper/main.pdf` for download. After downloading, unzip into the repo's root on your local machine so the next iteration of analysis / writeup sees the data.

In [ ]:
import shutil, os
from pathlib import Path
out = Path('gnnmarl_outputs.zip')
if out.exists(): out.unlink()
# Bundle the parts a downstream user actually needs: result CSVs + figures + PDF.
files = []
for p in Path('results').rglob('*'):
    if p.is_file(): files.append(p)
for p in Path('paper').glob('main.pdf'):
    files.append(p)
for p in Path('paper').glob('references.bib'):
    files.append(p)
for p in Path('paper/results').glob('*.tex'):
    files.append(p)
for p in Path('paper/figures').glob('*'):
    files.append(p)
import zipfile
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in files:
        zf.write(p)
print(f'wrote {out} -- {out.stat().st_size / 1e6:.1f} MB, {len(files)} files')
from google.colab import files as gcf
gcf.download(str(out))


## After downloading: integration on your local machine

```bash
cd path/to/when-graphs-help-marl
unzip gnnmarl_outputs.zip   # overlays results/ and paper/
open paper/main.pdf
```

If you want to regenerate the paper from the CSVs on your local machine:

```bash
bash scripts/build_paper.sh
```